
# DataSpark Cleansing Module — Notebook de ejemplos

Este notebook muestra **cada función principal** del módulo `dataspark.cleansing` con un dataset real y ligero obtenido de Internet.

Dataset usado: **Titanic (Seaborn mirror en GitHub)**.
- URL: `https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv`


In [ ]:

import pandas as pd
import numpy as np

from dataspark.cleansing import DataCleaner, Deduplicator, OutlierDetector, TypeInferenceEngine

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)


## 1) Cargar una base de datos ligera desde Internet


In [ ]:
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"

# Intento principal: dataset ligero desde Internet
try:
    raw_df = pd.read_csv(url)
    source_name = f"Titanic (URL): {url}"
except Exception as exc:
    # Fallback para entornos sin salida a internet
    from sklearn.datasets import load_iris
    iris = load_iris(as_frame=True)
    raw_df = iris.frame
    source_name = "Fallback local: sklearn.datasets.load_iris()"
    print("No se pudo descargar la URL por restricciones de red:", exc)

print("Fuente:", source_name)
print("Shape original:", raw_df.shape)
raw_df.head()


## 2) Crear una versión de trabajo con ruido (nulos, duplicados y outliers)


In [ ]:

df = raw_df.copy()

rename_map = {
    "survived": " Survived ",
    "fare": "Fare($)",
    "embarked": "Embarked Port"
}
df = df.rename(columns=rename_map)

if "class" in df.columns:
    df["class"] = df["class"].astype(str).map(lambda x: f"  {x}  ")

n = len(df)
idx = np.random.default_rng(42).choice(n, size=30, replace=False)
df.loc[idx[:10], "Fare($)"] = np.nan
df.loc[idx[10:20], "age"] = np.nan
df.loc[idx[20:30], "Embarked Port"] = np.nan

out_idx = np.random.default_rng(7).choice(n, size=5, replace=False)
df.loc[out_idx, "Fare($)"] = df["Fare($)"].fillna(df["Fare($)"].median()).max() * 15

rows_to_duplicate = df.sample(5, random_state=123)
df = pd.concat([df, rows_to_duplicate], ignore_index=True)

print("Shape con ruido:", df.shape)
df.head()


## 3) `DataCleaner` — todas sus funciones


In [ ]:

cleaner = DataCleaner(missing_strategy="median", standardize_columns=True, strip_whitespace=True)

missing_profile = cleaner.profile_missing(df)
missing_profile.head(10)


In [ ]:

cleaner.fit(df)
df_clean = cleaner.transform(df)
print(df_clean.shape)
df_clean.head()


In [ ]:

df_clean_ft = DataCleaner(missing_strategy="mode").fit_transform(df)
print(df_clean_ft.shape)


In [ ]:

helper_cleaner = DataCleaner(missing_strategy="median")

demo = df.head(8).copy()
print("Columnas antes:", demo.columns.tolist())
std_demo = helper_cleaner._standardize_columns(demo.copy())
print("Columnas después de _standardize_columns:", std_demo.columns.tolist())

strip_demo = helper_cleaner._strip_whitespace(std_demo.copy())
print(strip_demo[[c for c in strip_demo.columns if c in ["class"]]].head())

helper_cleaner.fit(std_demo)
handled_demo = helper_cleaner._handle_missing(std_demo.copy())
print("Nulos tras _handle_missing:", handled_demo.isna().sum().sum())

knn_demo = DataCleaner(missing_strategy="knn", knn_neighbors=3)._knn_impute(std_demo.copy())
print("Resultado _knn_impute:", knn_demo.shape)


## 4) `Deduplicator` — todas sus funciones


In [ ]:

dedup_exact = Deduplicator(strategy="exact")

dups_exact = dedup_exact.find_duplicates(df_clean)
print("Duplicados exactos encontrados:", len(dups_exact))

df_no_dups = dedup_exact.deduplicate(df_clean)
print("Filas después de deduplicate:", len(df_no_dups))

hashes = dedup_exact.hash_rows(df_no_dups.head(5))
hashes


In [ ]:

fuzzy_df = pd.DataFrame({
    "name": ["Juan Perez", "Juan Peres", "Ana Gómez", "Ana Gomez", "Carlos Ruiz"],
    "city": ["CDMX", "CDMX", "Bogotá", "Bogota", "Lima"]
})

dedup_fuzzy = Deduplicator(strategy="fuzzy", similarity_threshold=0.80, subset=["name", "city"])
print("_char_similarity ejemplo:", dedup_fuzzy._char_similarity("Juan Perez", "Juan Peres"))
print("Máscara _fuzzy_duplicates:")
print(dedup_fuzzy._fuzzy_duplicates(fuzzy_df))

print("Duplicados fuzzy:")
print(dedup_fuzzy.find_duplicates(fuzzy_df))


## 5) `OutlierDetector` — todas sus funciones


In [ ]:

out_df = df_clean.copy()
num_cols = [c for c in ["fare", "age"] if c in out_df.columns]
out_df[num_cols].describe()


In [ ]:

od = OutlierDetector(method="iqr", threshold=1.5)
mask = od.detect(out_df, columns=["fare"])
print(mask["fare"].value_counts())

removed = od.remove(out_df, columns=["fare"])
print("Filas tras remove:", removed.shape)

capped = od.cap(out_df, columns=["fare"])
print("Máx original vs capped:", out_df["fare"].max(), capped["fare"].max())


In [ ]:

s = out_df["fare"]
print("_iqr outliers:", od._iqr(s).sum())

odz = OutlierDetector(method="zscore", threshold=3.0)
print("_zscore outliers:", odz._zscore(s).sum())

odm = OutlierDetector(method="mad", threshold=3.5)
print("_mad outliers:", odm._mad(s).sum())

odi = OutlierDetector(method="isolation_forest", contamination=0.02)
print("_iforest outliers:", odi._iforest(s).sum())


## 6) `TypeInferenceEngine` — todas sus funciones


In [ ]:

tie_df = df_no_dups.copy()

tie_df["bool_like"] = np.where(np.arange(len(tie_df)) % 2 == 0, "1", "0")
tie_df["date_like"] = pd.date_range("2024-01-01", periods=len(tie_df), freq="D").astype(str)
tie_df["numeric_like"] = np.where(np.arange(len(tie_df)) % 3 == 0, "10", "20")

engine = TypeInferenceEngine(categorical_threshold=0.1)

report_before = engine.report(tie_df)
report_before.head(10)


In [ ]:

converted = engine.infer_and_convert(tie_df)
converted.dtypes.head(15)


In [ ]:

print("_suggest_dtype(bool_like):", engine._suggest_dtype(tie_df["bool_like"]))
print("_suggest_dtype(date_like):", engine._suggest_dtype(tie_df["date_like"]))
print("_suggest_dtype(numeric_like):", engine._suggest_dtype(tie_df["numeric_like"]))

sample_numeric = pd.Series([1, 2, 3, 4], dtype="int64")
print("dtype antes:", sample_numeric.dtype)
print("dtype después _downcast_numeric:", engine._downcast_numeric(sample_numeric).dtype)

print("_convert_column(date_like) dtype:", engine._convert_column(tie_df["date_like"]).dtype)



## 7) Conclusiones rápidas

- El notebook cubre todas las funcionalidades clave del módulo de limpieza.
- Usa un dataset real y liviano descargado de Internet.
- Incluye tanto API pública como métodos internos (educativo/debug).
